# High Frequency Trading Model with Deep Reinforcement Learning

This notebook implements a sophisticated high-frequency trading model using Deep Q-Learning (DQN) with Interactive Brokers data. The model combines modern deep learning techniques with traditional technical analysis to make trading decisions during London and New York sessions.

## Project Overview
- **Architecture**: Double DQN with experience replay and target networks
- **Features**: Advanced technical indicators and session-based trading
- **Trading Windows**: London (07:00-16:00 UTC) and New York (13:00-22:00 UTC)
- **Risk Management**: Integrated position sizing and drawdown control
- **Performance Tracking**: Session-specific metrics and early stopping

## Dependencies and Setup
The model utilizes several key libraries:
- **PyTorch**: Deep learning framework for neural network implementation
- **Pandas-TA**: Technical analysis library for market indicators
- **Gym**: Reinforcement learning environment framework
- **Seaborn/Matplotlib**: Visualization tools for performance analysis

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import pandas_ta as ta
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random
import logging
import os
from pathlib import Path
from tqdm.auto import tqdm
import gym
from gym import spaces
from typing import Tuple, Dict, List
from datetime import datetime, time
from collections import defaultdict

# Add visualization imports
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style and configuration
sns.set_style('darkgrid')
plt.style.use('default')
%matplotlib inline

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## Configuration
### Model Parameters
The reinforcement learning configuration includes carefully tuned hyperparameters:
- Learning rate: Controls model adaptation speed
- Gamma (discount factor): Balances immediate vs future rewards
- Epsilon parameters: Manages exploration vs exploitation
- Batch size: Optimizes training stability

### Technical Indicators
The model incorporates multiple technical indicators for market analysis:
1. **Trend Indicators**
   - MACD (Moving Average Convergence Divergence)
   - ADX (Average Directional Index)
   
2. **Momentum Indicators**
   - RSI (Relative Strength Index)
   - CCI (Commodity Channel Index)
   
3. **Volatility Measures**
   - Bollinger Bands
   - ATR (Average True Range)
   
4. **Market Structure**
   - VHF (Vertical Horizontal Filter)
   - ERI (Elder Ray Index)

In [ ]:
# Project paths
BASE_DIR = Path().absolute()
DATA_DIR = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "models"

# Create directories if they don't exist
MODELS_DIR.mkdir(exist_ok=True)

# Define timeframe date ranges
TIMEFRAME_RANGES = {
    'M5': {'start': '2023-11-21', 'end': '2025-03-25'},
    'M15': {'start': '2021-03-23', 'end': '2025-03-25'},
    'M30': {'start': '2017-03-16', 'end': '2025-03-25'},
    'H1': {'start': '2009-03-13', 'end': '2025-03-25'},
    'H4': {'start': '2009-03-13', 'end': '2025-03-25'}
}

# Trading sessions in UTC
TRADING_SESSIONS = {
    'London': {
        'start': time(7, 0),   # 7:00 UTC (8:00 London)
        'end': time(16, 0)     # 16:00 UTC (17:00 London)
    },
    'NewYork': {
        'start': time(13, 0),  # 13:00 UTC (8:00 NY)
        'end': time(22, 0)     # 22:00 UTC (17:00 NY)
    }
}

# RL Configuration
RL_CONFIG = {
    'learning_rate': 0.0003,     # Reduced learning rate
    'gamma': 0.95,              # Slightly reduced discount factor
    'epsilon_start': 1.0,
    'epsilon_end': 0.05,        # Higher minimum exploration
    'epsilon_decay': 0.997,     # Slower epsilon decay
    'batch_size': 64,           # Larger batch size
    'min_memory_size': 1000,    # Minimum samples before training
    'reward_scaling': 0.1,      # Scale rewards for better stability
    'target_update_freq': 5,    # Target network update frequency
    'early_stopping_patience': 15  # Episodes without improvement before stopping
}

# Technical Indicators Configuration
TECHNICAL_INDICATORS = {
    'RSI': {'length': 14},
    'MACD': {'fast': 12, 'slow': 26, 'signal': 9},
    'BB': {'length': 20, 'std': 2},
    'ATR': {'length': 14},
    'CCI': {'length': 20},  # Added Commodity Channel Index
    'VHF': {'length': 28},  # Added Vertical Horizontal Filter
    'ERI': {'length': 13},  # Added Elder Ray Index
    'ADX': {'length': 14}   # Added Average Directional Index
}

## Data Processing Pipeline
The DataProcessor class implements a robust data handling system:

1. **Data Loading**
   - Efficient chunk-based reading for large datasets
   - Automatic data type conversion
   - Time series indexing and validation

2. **Data Cleaning**
   - Handles missing values
   - Removes duplicates
   - Ensures data consistency

3. **Feature Engineering**
   - Calculates technical indicators
   - Normalizes features
   - Creates derived signals

In [ ]:
class DataProcessor:
    def __init__(self):
        self.logger = logging.getLogger(__name__)
        self.sessions = TRADING_SESSIONS
        print("\n[DataProcessor] Initialized with trading sessions")

    def _is_active_session(self, timestamp) -> dict:
        """Check if timestamp is within trading sessions"""
        try:
            current_time = timestamp.time()
            return {
                'London': self.sessions['London']['start'] <= current_time <= self.sessions['London']['end'],
                'NewYork': self.sessions['NewYork']['start'] <= current_time <= self.sessions['NewYork']['end']
            }
        except AttributeError:
            return {'London': False, 'NewYork': False}

    def load_data(self, filepath: str) -> pd.DataFrame:
        try:
            print(f"\n{'='*50}")
            print(f"[DataProcessor] Loading data from {filepath}")
            
            # Determine timeframe from filename
            timeframe = Path(filepath).stem
            date_range = TIMEFRAME_RANGES.get(timeframe)

            if not date_range:
                raise ValueError(f"Unknown timeframe: {timeframe}")

            # Define expected columns
            expected_columns = ['Time', 'Open', 'High', 'Low', 'Close', 'Volume']
            
            # Show progress while reading large files with correct delimiter
            print("\n[DataProcessor] Reading data chunks:")
            chunks = pd.read_csv(
                filepath,
                header=None,
                names=expected_columns,
                sep=',',  # Changed to comma separator
                parse_dates=['Time'],  # Parse dates during reading
                date_parser=lambda x: pd.to_datetime(x, format='%Y-%m-%d %H:%M'),
                chunksize=1000
            )
            
            df_chunks = []
            for chunk in tqdm(chunks, desc="Reading data", ncols=100):
                df_chunks.append(chunk)
            df = pd.concat(df_chunks)
            
            print(f"\n[DataProcessor] Raw data shape: {df.shape}")
            initial_total = len(df)
            
            if df.empty:
                raise ValueError("Empty dataframe loaded")
            
            print("\n[DataProcessor] Processing steps:")
            
            # Filter by date range
            print("1. Filtering by date range...")
            df = df[
                (df['Time'] >= pd.to_datetime(date_range['start'])) & 
                (df['Time'] <= pd.to_datetime(date_range['end']))
            ]
            print(f"Rows after time filtering: {len(df)} (dropped {initial_total - len(df)} rows)")
            
            # Numeric conversion - more selective handling
            print("2. Converting numeric columns...")
            numeric_columns = ['Open', 'High', 'Low', 'Close', 'Volume']
            for col in numeric_columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                if col != 'Volume':
                    df = df.dropna(subset=[col])
                else:
                    df[col] = df[col].fillna(0)
            
            print(f"Rows after numeric conversion: {len(df)} (dropped {initial_total - len(df)} rows)")
            
            # Data cleaning and indexing
            print("3. Setting index and cleaning data...")
            df = df.set_index('Time')
            df = df[~df.index.duplicated(keep='first')]
            df = df.sort_index()
            
            # Add trading session flags
            print("4. Adding trading session information...")
            df['Hour'] = df.index.hour
            
            # Use vectorized operations for session flags
            london_start = self.sessions['London']['start'].hour
            london_end = self.sessions['London']['end'].hour
            newyork_start = self.sessions['NewYork']['start'].hour
            newyork_end = self.sessions['NewYork']['end'].hour
            
            df['London_Session'] = (df['Hour'] >= london_start) & (df['Hour'] <= london_end)
            df['NewYork_Session'] = (df['Hour'] >= newyork_start) & (df['Hour'] <= newyork_end)
            df['Overlap_Session'] = df['London_Session'] & df['NewYork_Session']
            df['Trading_Session'] = df['London_Session'] | df['NewYork_Session']
            
            # Additional time-based features
            df['Hour_of_Day'] = df.index.hour
            df['Day_of_Week'] = df.index.dayofweek
            
            # Drop temporary column
            df = df.drop('Hour', axis=1)
            
            # Print summary
            print(f"\n[DataProcessor] Data loading completed:")
            print(f" - Initial rows: {initial_total}")
            print(f" - Final rows: {len(df)}")
            print(f" - Date range: {df.index.min()} to {df.index.max()}")
            print(f" - Data retention: {(len(df)/initial_total)*100:.1f}%")
            print(f" - Columns: {', '.join(df.columns)}")
            print(f"{'-'*50}")
            
            return df
            
        except Exception as e:
            print(f"[DataProcessor] Error loading data: {str(e)}")
            raise

    def add_technical_indicators(self, df: pd.DataFrame) -> pd.DataFrame:
        try:
            print(f"\n{'='*50}")
            print("[DataProcessor] Adding technical indicators")
            initial_rows = len(df)
            
            # Initialize indicator strategy
            print("\n[DataProcessor] Setting up indicators:")
            for name, params in TECHNICAL_INDICATORS.items():
                print(f" - {name}: {params}")
            
            custom_strategy = ta.Strategy(
                name="custom_strategy",
                ta=[
                    {"kind": "rsi", "length": TECHNICAL_INDICATORS['RSI']['length']},
                    {"kind": "macd", "fast": TECHNICAL_INDICATORS['MACD']['fast'],
                     "slow": TECHNICAL_INDICATORS['MACD']['slow'],
                     "signal": TECHNICAL_INDICATORS['MACD']['signal']},
                    {"kind": "bbands", "length": TECHNICAL_INDICATORS['BB']['length'],
                     "std": TECHNICAL_INDICATORS['BB']['std']},
                    {"kind": "atr", "length": TECHNICAL_INDICATORS['ATR']['length']},
                    {"kind": "cci", "length": TECHNICAL_INDICATORS['CCI']['length']},
                    {"kind": "vhf", "length": TECHNICAL_INDICATORS['VHF']['length']},
                    {"kind": "eri", "length": TECHNICAL_INDICATORS['ERI']['length']},
                    {"kind": "adx", "length": TECHNICAL_INDICATORS['ADX']['length']}
                ]
            )
            
            # Store initial columns for comparison
            initial_columns = set(df.columns)
            
            # Calculate all indicators
            print("\n[DataProcessor] Calculating indicators...")
            df.ta.strategy(custom_strategy)
            
            # Print summary of added indicators
            new_columns = set(df.columns) - initial_columns
            print("\n[DataProcessor] Technical indicators added:")
            for col in sorted(new_columns):
                print(f" ✓ {col}")
            
            # Forward fill NaN values for indicators instead of dropping
            indicator_columns = list(new_columns)
            df[indicator_columns] = df[indicator_columns].ffill()
            
            # Only drop rows if all indicators are NaN
            df = df.dropna(subset=indicator_columns, how='all')
            
            print(f"\n[DataProcessor] Final statistics:")
            print(f" - Initial rows: {initial_rows}")
            print(f" - Final rows: {len(df)}")
            print(f" - Data retention: {(len(df)/initial_rows)*100:.1f}%")
            print(f" - Added indicators: {len(new_columns)}")
            print(f" - Total features: {len(df.columns)}")
            print(f"{'-'*50}")
            
            return df
            
        except Exception as e:
            print(f"[DataProcessor] Error in technical analysis: {str(e)}")
            raise

## Neural Network Architecture
The DQN implementation features:

1. **Network Structure**
   - Dynamic layer sizing based on input dimensions
   - Layer Normalization for training stability
   - Dropout layers to prevent overfitting
   - Xavier/Glorot initialization

2. **Advanced Features**
   - Double DQN architecture to reduce overestimation
   - Target network for stable learning
   - Gradient clipping to prevent exploding gradients

In [ ]:
class DQNNetwork(nn.Module):
    def __init__(self, input_size, output_size):
        super(DQNNetwork, self).__init__()
        
        # Validate input size
        if input_size <= 0:
            raise ValueError(f"Invalid input size: {input_size}")
            
        # Enhanced architecture with deeper layers
        h1_size = max(128, input_size * 2)
        h2_size = max(64, input_size)
        h3_size = max(32, input_size // 2)
        
        self.layers = nn.Sequential(
            nn.Linear(input_size, h1_size),
            nn.LayerNorm(h1_size),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(h1_size, h2_size),
            nn.LayerNorm(h2_size),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(h2_size, h3_size),
            nn.LayerNorm(h3_size),
            nn.ReLU(),
            nn.Dropout(0.1),
            
            nn.Linear(h3_size, output_size)
        )
        
        # Initialize weights using Xavier/Glorot initialization
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.constant_(m.bias, 0)
                
    def forward(self, x):
        # Ensure input tensor has correct shape
        if x.dim() == 1:
            x = x.unsqueeze(0)
        elif x.dim() == 3:
            x = x.squeeze(1)
        
        # Ensure float32 dtype
        x = x.float()
        
        # Validate input shape before forward pass
        batch_size, features = x.shape
        if features != self.layers[0].in_features:
            raise ValueError(f"Input features {features} doesn't match network input size {self.layers[0].in_features}")
        
        return self.layers(x)

In [ ]:
class ExperienceBuffer:
    def __init__(self, buffer_size=100000):
        self.buffer_size = buffer_size
        self.states = np.zeros((buffer_size, 0), dtype=np.float32)  # Dynamically sized
        self.actions = np.zeros(buffer_size, dtype=np.int64)
        self.rewards = np.zeros(buffer_size, dtype=np.float32)
        self.next_states = np.zeros((buffer_size, 0), dtype=np.float32)
        self.dones = np.zeros(buffer_size, dtype=np.float32)
        self.pos = 0
        self.size = 0

    def add(self, state, action, reward, next_state, done):
        # Resize arrays if necessary
        if self.states.shape[1] != len(state):
            state_dim = len(state)
            self.states = np.zeros((self.buffer_size, state_dim), dtype=np.float32)
            self.next_states = np.zeros((self.buffer_size, state_dim), dtype=np.float32)

        # Store experience
        self.states[self.pos] = state
        self.actions[self.pos] = action
        self.rewards[self.pos] = reward
        self.next_states[self.pos] = next_state
        self.dones[self.pos] = done

        self.pos = (self.pos + 1) % self.buffer_size
        self.size = min(self.size + 1, self.buffer_size)

    def sample(self, batch_size):
        indices = np.random.choice(self.size, batch_size, replace=False)
        return (
            torch.from_numpy(self.states[indices]),
            torch.from_numpy(self.actions[indices]),
            torch.from_numpy(self.rewards[indices]),
            torch.from_numpy(self.next_states[indices]),
            torch.from_numpy(self.dones[indices])
        )

    def clear(self):
        """Clear all buffer arrays"""
        self.states = np.zeros((self.buffer_size, 0), dtype=np.float32)
        self.next_states = np.zeros((self.buffer_size, 0), dtype=np.float32)
        self.actions = np.zeros(self.buffer_size, dtype=np.int64)
        self.rewards = np.zeros(self.buffer_size, dtype=np.float32)
        self.dones = np.zeros(self.buffer_size, dtype=np.float32)
        self.pos = 0
        self.size = 0

class TradingAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = ExperienceBuffer(100000)
        self.batch_size = RL_CONFIG['batch_size']
        
        self.gamma = RL_CONFIG['gamma']
        self.epsilon = RL_CONFIG['epsilon_start']
        self.epsilon_min = RL_CONFIG['epsilon_end']
        self.epsilon_decay = RL_CONFIG['epsilon_decay']
        
        # Initialize models
        self.model = DQNNetwork(state_size, action_size)
        self.target_model = DQNNetwork(state_size, action_size)
        self.target_model.load_state_dict(self.model.state_dict())
        
        # Use learning rate scheduler without verbose parameter
        self.optimizer = optim.Adam(self.model.parameters(), lr=RL_CONFIG['learning_rate'])
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, 
            mode='max', 
            factor=0.5, 
            patience=10
        )
        
        # Initialize replay memory with random experiences
        self.min_memory_size = 1000
        self.update_target_every = 5  # Update target network every N episodes
        self.episode_count = 0
        self.loss_history = []
        self.warmup_buffer = []  # Temporary buffer for warmup
        self.warmup_in_progress = False
        
    def update_target_model(self):
        self.target_model.load_state_dict(self.model.state_dict())
    
    def warmup_memory(self, env, num_actions=1000):
        """Pre-fill replay memory with random actions using batch processing"""
        if self.warmup_in_progress:
            logger.warning("Warmup already in progress, skipping...")
            return

        self.warmup_in_progress = True
        logger.info(f"Starting warmup process targeting {num_actions} actions...")

        # Clear existing memory before warmup
        self.memory.clear()
        self.warmup_buffer.clear()

        try:
            state = env.reset()
            logger.info(f"Initial state shape: {state.shape}")

            experiences = []
            episode_count = 0
            steps_in_episode = 0
            total_steps = 0
            max_steps = num_actions * 2  # Timeout protection

            with tqdm(total=num_actions, desc="Warmup Progress") as pbar:
                while len(experiences) < num_actions and total_steps < max_steps:
                    if steps_in_episode > 1000:  # Safety check for infinite loops
                        logger.warning(f"Episode {episode_count} exceeded 1000 steps. Resetting...")
                        state = env.reset()
                        steps_in_episode = 0
                        episode_count += 1
                        continue

                    action = random.randrange(self.action_size)
                    try:
                        next_state, reward, done, info = env.step(action)
                        total_steps += 1
                        steps_in_episode += 1

                        # Validate state and next_state
                        if not isinstance(next_state, np.ndarray):
                            raise ValueError(f"Invalid next_state type: {type(next_state)}")
                        if next_state.shape != state.shape:
                            raise ValueError(f"State shape mismatch: {state.shape} vs {next_state.shape}")

                        experiences.append((state, action, reward, next_state, done))
                        pbar.update(1)

                        if len(experiences) % 100 == 0:
                            logger.info(f"Collected {len(experiences)}/{num_actions} experiences ")
                            logger.info(f"Current episode: {episode_count}, Steps in episode: {steps_in_episode}")

                        if done:
                            logger.debug(f"Episode {episode_count} completed after {steps_in_episode} steps")
                            state = env.reset()
                            steps_in_episode = 0
                            episode_count += 1
                        else:
                            state = next_state

                    except Exception as e:
                        logger.error(f"Error during step: {str(e)}")
                        logger.error(f"Action: {action}, State shape: {state.shape}")
                        raise

            # Process experiences in batches
            logger.info("Processing warmup experiences...")
            batch_size = 100
            for i in range(0, len(experiences), batch_size):
                batch = experiences[i:i + batch_size]
                for exp in batch:
                    self.memory.add(*exp)
                logger.info(f"Processed {min(i + batch_size, len(experiences))}/{len(experiences)} experiences")

            logger.info(f"Warmup completed:")
            logger.info(f" - Total episodes: {episode_count}")
            logger.info(f" - Total steps: {total_steps}")
            logger.info(f" - Memory size: {self.memory.size}")
            logger.info(f" - Average steps per episode: {total_steps/(episode_count or 1):.1f}")

            if total_steps >= max_steps:
                logger.warning(f"Warmup timed out after {total_steps} steps")

        except Exception as e:
            logger.error(f"Warmup failed: {str(e)}")
            logger.error("Clearing memory due to error...")
            self.memory.clear()
            raise
        finally:
            self.warmup_in_progress = False

    def remember(self, state, action, reward, next_state, done):
        self.memory.add(state, action, reward, next_state, done)

    def act(self, state, training=True):
        state = np.asarray(state, dtype=np.float32)
        if training and (random.random() < self.epsilon):
            return random.randrange(self.action_size)
        
        # Ensure proper state shape and type
        state = torch.FloatTensor(state)
        if state.dim() == 1:
            state = state.unsqueeze(0)
        
        # Debug dimensions
        if self.model.layers[0].in_features != state.shape[-1]:
            raise ValueError(f"State features {state.shape[-1]} doesn't match model input {self.model.layers[0].in_features}")
            
        with torch.no_grad():
            action_values = self.model(state)
        return torch.argmax(action_values).item()

    def train(self, episode_reward=None):
        if self.memory.size < self.min_memory_size:
            return

        # Get batch of experiences efficiently
        states, actions, rewards, next_states, dones = self.memory.sample(self.batch_size)

        # Current Q values - more efficient gathering
        current_q_values = self.model(states).gather(1, actions.unsqueeze(1))

        # Next Q values with Double DQN - use vectorized operations
        with torch.no_grad():
            next_actions = self.model(next_states).max(1)[1]
            next_q_values = self.target_model(next_states)
            next_q_values = next_q_values.gather(1, next_actions.unsqueeze(1)).squeeze(1)

        # Compute target values in single operation
        target_q_values = rewards + (1 - dones) * self.gamma * next_q_values

        # Huber loss for better stability
        loss = nn.SmoothL1Loss()(current_q_values.squeeze(), target_q_values)
        
        # Optimize
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
        self.optimizer.step()
        
        # Update target network periodically
        if self.episode_count % self.update_target_every == 0:
            self.update_target_model()
        
        # Update learning rate if reward is provided
        if episode_reward is not None:
            self.scheduler.step(episode_reward)
        
        # Decay epsilon
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
        
        # Store loss for monitoring
        self.loss_history.append(loss.item())
        return loss.item()

## Trading Environment
The ForexTradingEnv class provides a gym-compatible trading simulation:

1. **State Space**
   - Technical indicators
   - Position information
   - Market data features

2. **Action Space**
   - Buy (Long)
   - Sell (Short)
   - Hold

3. **Reward System**
   - Profit/Loss based rewards
   - Risk-adjusted returns
   - Trading frequency penalties

In [ ]:
class ForexTradingEnv(gym.Env):
    def __init__(self, df: pd.DataFrame, validation_mode=False):
        super(ForexTradingEnv, self).__init__()
        
        self.df = df
        self.validation_mode = validation_mode
        self.current_step = 0
        self.position = None
        self.entry_time = None
        self.entry_price = None
        
        # Trading metrics
        self.total_pnl = 0
        self.max_drawdown = 0
        self.current_drawdown = 0
        self.peak_value = 0
        self.trades_history = []
        
        # Action and observation spaces
        self.action_space = spaces.Discrete(3)  # Buy, Sell, Hold
        
        # Ensure feature columns only include numeric data
        self.feature_columns = df.select_dtypes(include=[np.number]).columns
        # Calculate state dimension including position flags and metrics
        self.state_dim = len(self.feature_columns) + 3  # +3 for position flags and metrics
        
        # Update observation space with correct dimensions
        self.observation_space = spaces.Box(
            low=-np.inf, 
            high=np.inf,
            shape=(self.state_dim,),
            dtype=np.float32
        )

    def _is_trading_allowed(self) -> bool:
        """Check if trading is allowed in current session"""
        current_data = self.df.iloc[self.current_step]
        return current_data['Trading_Session']

    def _calculate_reward(self, price_change: float) -> float:
        """Calculate scaled reward with additional factors"""
        current_data = self.df.iloc[self.current_step]
        base_reward = price_change * 100  # Convert to percentage
        
        # Scale reward based on session
        if current_data['Overlap_Session']:
            base_reward *= 1.2  # 20% bonus during session overlap
        elif not current_data['Trading_Session']:
            base_reward *= 0.5  # 50% penalty outside trading sessions
        
        # Add drawdown penalty
        if self.current_drawdown < self.max_drawdown:
            self.max_drawdown = self.current_drawdown
            base_reward *= 0.8  # Penalty for new drawdown
        
        return base_reward * RL_CONFIG['reward_scaling']

    def step(self, action: int) -> Tuple[np.array, float, bool, Dict]:
        current_data = self.df.iloc[self.current_step]
        current_price = current_data['Close']
        reward = 0
        
        # Execute trading action only during trading sessions
        if self._is_trading_allowed():
            if action == 0 and self.position is None:  # Buy
                self.position = 'long'
                self.entry_price = current_price
                self.entry_time = current_data.name
            elif action == 1 and self.position is None:  # Sell
                self.position = 'short'
                self.entry_price = current_price
                self.entry_time = current_data.name
        
        # Move to next step
        self.current_step += 1
        done = self.current_step >= len(self.df) - 1
        
        # Calculate position outcome
        if self.position is not None:
            next_data = self.df.iloc[self.current_step]
            next_price = next_data['Close']
            price_change = (next_price - self.entry_price) / self.entry_price
            
            # Calculate reward
            if self.position == 'long':
                reward = self._calculate_reward(price_change)
            else:  # short position
                reward = self._calculate_reward(-price_change)
            
            # Update metrics
            self.total_pnl += reward
            self.current_drawdown = min(0, self.total_pnl - self.peak_value)
            self.peak_value = max(self.peak_value, self.total_pnl)
            
            # Record trade
            self.trades_history.append({
                'position': self.position,
                'entry_time': self.entry_time,
                'exit_time': next_data.name,
                'entry_price': self.entry_price,
                'exit_price': next_price,
                'reward': reward,
                'pnl': self.total_pnl,
                'drawdown': self.current_drawdown,
                'London_Session': next_data['London_Session'],
                'NewYork_Session': next_data['NewYork_Session'],
                'Overlap_Session': next_data['Overlap_Session'],
                'duration': (next_data.name - self.entry_time).seconds / 3600  # duration in hours
            })
            
            # Reset position
            self.position = None
            self.entry_price = None
            self.entry_time = None
        
        return self._get_state(), reward, done, {
            'trades': len(self.trades_history),
            'total_pnl': self.total_pnl,
            'max_drawdown': self.max_drawdown,
            'current_drawdown': self.current_drawdown,
            'win_rate': sum(1 for t in self.trades_history if t['reward'] > 0) / max(1, len(self.trades_history)),
            'trade_info': self.trades_history[-1] if self.trades_history else None
        }

    def reset(self):
        self.current_step = 0
        self.position = None
        self.entry_price = None
        self.entry_time = None
        self.trades_history = []
        self.total_pnl = 0
        self.max_drawdown = 0
        self.current_drawdown = 0
        self.peak_value = 0
        return self._get_state()

    def _get_state(self) -> np.array:
        """Create the state vector by combining features"""
        current_data = self.df.iloc[self.current_step]
        
        # Get numeric features and ensure float32
        features = current_data[self.feature_columns].values.astype(np.float32)
        
        # Add position and metrics (ensure consistent with state_dim)
        position_flag = np.zeros(1, dtype=np.float32)  # Simplified to single flag
        if self.position == 'long':
            position_flag[0] = 1
        elif self.position == 'short':
            position_flag[0] = -1
            
        state = np.concatenate([
            features,
            position_flag,
            [float(self.total_pnl), float(self.current_drawdown)]
        ]).astype(np.float32)
        
        return state

## Training Process
The training implementation includes:

1. **Setup Phase**
   - Data preprocessing
   - Environment initialization
   - Agent configuration

2. **Training Loop**
   - Experience collection
   - Batch learning
   - Model optimization

3. **Model Selection**
   - Performance tracking
   - Best model preservation
   - Multiple timeframe handling

In [ ]:
def setup_training(timeframe: str):
    """Initialize training components for a specific timeframe"""
    # Load and process data
    data_file = DATA_DIR / f"{timeframe}.csv"
    processor = DataProcessor()
    df = processor.load_data(data_file)
    
    # Add debug logging
    logger.info(f"Original dataframe shape: {df.shape}")
    logger.info(f"Original columns: {df.columns.tolist()}")
    
    # Process indicators
    df = processor.add_technical_indicators(df)
    df = df.dropna()  # Remove rows with NaN values
    
    # Validate processed data
    if len(df) < 30:  # Minimum required rows
        raise ValueError(f"Insufficient data for {timeframe}. Need at least 30 rows, got {len(df)}")
    
    # Split data into training and validation sets (80-20 split)
    split_idx = int(len(df) * 0.8)
    train_df = df[:split_idx].copy()
    val_df = df[split_idx:].copy()
    
    # Log split info
    logger.info(f"Training set size: {len(train_df)}, Validation set size: {len(val_df)}")
    
    # Create training and validation environments
    train_env = ForexTradingEnv(train_df, validation_mode=False)
    val_env = ForexTradingEnv(val_df, validation_mode=True)
    logger.info(f"Environment state dimension: {train_env.state_dim}")
    
    # Initialize agent with correct dimensions
    agent = TradingAgent(state_size=train_env.state_dim, action_size=train_env.action_space.n)
    logger.info(f"Agent initialized with state_size={train_env.state_dim}, action_size={train_env.action_space.n}")
    
    return train_env, val_env, agent, train_df

In [ ]:
class TradingMetrics:
    @staticmethod
    def calculate_metrics(trades_history: List[Dict], episode_rewards: List[float]) -> Dict:
        """Calculate comprehensive trading metrics"""
        if not trades_history:
            return {}

        # Session-specific metrics
        session_trades = defaultdict(list)
        for trade in trades_history:
            if trade.get('London_Session'):
                session_trades['London'].append(trade)
            if trade.get('NewYork_Session'):
                session_trades['NewYork'].append(trade)
            if trade.get('Overlap_Session'):
                session_trades['Overlap'].append(trade)

        metrics = {
            # Overall performance
            'total_reward': sum(episode_rewards),
            'mean_reward': np.mean(episode_rewards),
            'reward_std': np.std(episode_rewards),
            'max_drawdown': TradingMetrics._calculate_max_drawdown(episode_rewards),
            
            # Trading metrics
            'total_trades': len(trades_history),
            'win_rate': sum(1 for t in trades_history if t['reward'] > 0) / len(trades_history),
            'profit_factor': TradingMetrics._calculate_profit_factor(trades_history),
            'avg_trade_duration': np.mean([t['duration'] for t in trades_history if 'duration' in t]),
            
            # Session performance
            'session_metrics': {
                session: {
                    'trades': len(trades),
                    'win_rate': sum(1 for t in trades if t['reward'] > 0) / max(len(trades), 1),
                    'total_reward': sum(t['reward'] for t in trades),
                    'avg_reward': np.mean([t['reward'] for t in trades]) if trades else 0
                }
                for session, trades in session_trades.items()
            }
        }
        
        return metrics

    @staticmethod
    def _calculate_max_drawdown(rewards: List[float]) -> float:
        cumulative = np.cumsum(rewards)
        max_dd = 0
        peak = cumulative[0]
        
        for value in cumulative[1:]:
            if value > peak:
                peak = value
            dd = (peak - value) / peak if peak != 0 else 0
            max_dd = max(max_dd, dd)
        
        return max_dd

    @staticmethod
    def _calculate_profit_factor(trades: List[Dict]) -> float:
        winning_trades = sum(t['reward'] for t in trades if t['reward'] > 0)
        losing_trades = abs(sum(t['reward'] for t in trades if t['reward'] < 0))
        return winning_trades / losing_trades if losing_trades != 0 else float('inf')

def plot_training_metrics(history: List[Dict], timeframe: str):
    """Visualize training metrics"""
    plt.style.use('seaborn')
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    
    # Extract metrics
    episodes = [h['episode'] for h in history]
    rewards = [h['total_reward'] for h in history]
    win_rates = [h['metrics']['win_rate'] for h in history]
    
    # Plot rewards
    ax1.plot(episodes, rewards)
    ax1.set_title('Total Reward per Episode')
    ax1.set_xlabel('Episode')
    ax1.set_ylabel('Reward')
    
    # Plot win rates
    ax2.plot(episodes, win_rates)
    ax2.set_title('Win Rate per Episode')
    ax2.set_xlabel('Episode')
    ax2.set_ylabel('Win Rate')
    
    # Plot session win rates
    session_win_rates = {
        'London': [h['metrics']['session_metrics'].get('London', {}).get('win_rate', 0) for h in history],
        'NewYork': [h['metrics']['session_metrics'].get('NewYork', {}).get('win_rate', 0) for h in history],
        'Overlap': [h['metrics']['session_metrics'].get('Overlap', {}).get('win_rate', 0) for h in history]
    }
    
    for session, rates in session_win_rates.items():
        ax3.plot(episodes, rates, label=session)
    ax3.set_title('Session Win Rates')
    ax3.set_xlabel('Episode')
    ax3.set_ylabel('Win Rate')
    ax3.legend()
    
    # Plot drawdown
    drawdowns = [h['metrics'].get('max_drawdown', 0) for h in history]
    ax4.plot(episodes, drawdowns)
    ax4.set_title('Maximum Drawdown')
    ax4.set_xlabel('Episode')
    ax4.set_ylabel('Drawdown')
    
    plt.tight_layout()
    plt.savefig(f'training_metrics_{timeframe}.png')
    plt.close()

In [ ]:
def train_model(timeframe: str, episodes: int = 1000):
    """Train the model with early stopping and enhanced metrics"""
    # Setup phase
    train_env, val_env, agent, df = setup_training(timeframe)
    best_reward = float('-inf')
    patience = RL_CONFIG['early_stopping_patience']
    no_improvement = 0
    training_history = []
    episode_rewards = []

    try:
        # Warmup phase
        logger.info("Starting warmup...")
        agent.warmup_memory(train_env, num_actions=RL_CONFIG['min_memory_size'])
        logger.info("Warmup completed. Starting training...")

        # Main training loop
        for episode in range(episodes):
            logger.debug(f"Starting episode {episode + 1}")
            state = train_env.reset()
            episode_reward = 0
            done = False
            steps = 0

            # Episode loop
            while not done and steps < 10000:
                try:
                    action = agent.act(state)
                    next_state, reward, done, info = train_env.step(action)

                    agent.remember(state, action, reward, next_state, done)
                    loss = agent.train(episode_reward=episode_reward)

                    state = next_state
                    episode_reward += reward
                    steps += 1
                except Exception as e:
                    logger.error(f"Error in episode step: {str(e)}")
                    raise

            # After episode updates
            episode_rewards.append(episode_reward)
            if len(train_env.trades_history) > 0:
                metrics = TradingMetrics.calculate_metrics(
                    train_env.trades_history, [episode_reward])
            else:
                metrics = {'win_rate': 0, 'total_trades': 0}

            # Save history
            training_history.append({
                'episode': episode + 1,
                'total_reward': episode_reward,
                'metrics': metrics,
                'steps': steps
            })

            # Progress logging
            if (episode + 1) % 1 == 0:  # Log every episode
                logger.info(
                    f"Episode {episode + 1}/{episodes} - " +
                    f"Steps: {steps} - " +
                    f"Reward: {episode_reward:.2f} - " +
                    f"Trades: {metrics.get('total_trades', 0)}"
                )

            # Model saving
            if episode_reward > best_reward:
                best_reward = episode_reward
                model_path = MODELS_DIR / f"best_model_{timeframe}.pth"
                torch.save({
                    'episode': episode,
                    'model_state_dict': agent.model.state_dict(),
                    'optimizer_state_dict': agent.optimizer.state_dict(),
                    'metrics': metrics,
                    'hyperparameters': RL_CONFIG
                }, model_path)
                no_improvement = 0
                logger.info(f"New best model saved with reward: {best_reward:.2f}")
            else:
                no_improvement += 1

            # Early stopping
            if no_improvement >= patience:
                logger.info(f"Early stopping triggered after {patience} episodes")
                break

            # Update target network
            if episode % agent.update_target_every == 0:
                agent.update_target_model()

        logger.info("Training completed successfully")
        plot_training_metrics(training_history, timeframe)
        return training_history

    except Exception as e:
        logger.error(f"Training error: {str(e)}")
        logger.error("Traceback:", exc_info=True)
        raise

In [ ]:
# Train models for different timeframes
timeframes = ['M5', 'M15', 'M30', 'H1', 'H4']

for timeframe in timeframes:
    logger.info(f"Starting training for {timeframe} timeframe")
    try:
        history = train_model(timeframe)
        
        # Print final performance summary
        final_metrics = history[-1]['metrics']
        print(f"\nFinal Performance Summary for {timeframe}:")
        print(f"Total Trades: {final_metrics['total_trades']}")
        print(f"Overall Win Rate: {final_metrics['win_rate']:.2%}")
        print("\nSession Performance:")
        for session, metrics in final_metrics['session_metrics'].items():
            print(f"{session}:")
            print(f"  Trades: {metrics['trades']}")
            print(f"  Win Rate: {metrics['win_rate']:.2%}")
            print(f"  Avg Reward: {metrics['avg_reward']:.2f}")
    except Exception as e:
        logger.error(f"Error training {timeframe}: {str(e)}")
        continue

## Enhanced Warmup Logging
Added detailed logging and error handling during model warmup to track progress and diagnose issues.

In [ ]:
# Add exception handler for debugging
import sys
import traceback

def debug_exception_hook(exctype, value, tb):
    """Enhanced exception handler for better error tracking"""
    logger.error("Uncaught exception:")
    logger.error("Type: %s", exctype)
    logger.error("Value: %s", value)
    logger.error("Traceback:")
    for line in traceback.extract_tb(tb).format():
        logger.error(line)
    sys.__excepthook__(exctype, value, tb)  # Call the default handler

sys.excepthook = debug_exception_hook

# Configure more detailed logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler('trading_model.log')
    ]
)

## Setup Logging Configuration
Configure logging to prevent duplicate handlers and messages

In [ ]:
# Reset logging handlers to prevent duplicates
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Configure logging with a single handler
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levellevel)s - %(message)s',
    handlers=[
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

In [ ]:
# Debug and test run
print("Starting debug test run...")

# Reset logging handlers
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Configure single handler
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)

# Test run with debug info
timeframe = 'M5'
train_env, val_env, agent, df = setup_training(timeframe)

print("\nDebug Information:")
print(f"Memory size: {agent.memory.size}")
print(f"State shape: {train_env.reset().shape}")
print(f"Action space: {train_env.action_space.n}")
print(f"Feature columns: {len(train_env.feature_columns)}")

print("\nStarting training...")
history = train_model(timeframe)

In [ ]:
# Reset and configure logging
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Configure logging with proper format
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler()
    ]
)

# Test training with single timeframe
print("Starting training test...")
timeframe = 'M5'

# Debug info
train_env, val_env, agent, df = setup_training(timeframe)
print(f"\nEnvironment Info:")
print(f"State dimension: {train_env.state_dim}")
print(f"Action space: {train_env.action_space.n}")
print(f"Training data shape: {df.shape}")

# Run training
print("\nStarting training...")
history = train_model(timeframe, episodes=100)  # Reduced episodes for testing